# Q1 — Boxes

**Goal:** Write two functions in `src/boxes.py` that return the minimum number of boxes needed to store `n` coins, where **each box holds at most 30 coins**.

- `num_boxes(n: int) -> int`  
  *Non-recursive*.  
- `num_boxes_via_recursion(n: int) -> int`  
  *Same behaviour, implemented with recursion*.

**Assumptions:** `n` is a non-negative `int`.  
**Do not** include any testing code in `src/boxes.py`.

**Examples**

Example function calls:
```
>>> num_boxes(0)
0
>>> num_boxes(1)
1
>>> num_boxes(60)
2
>>> num_boxes_via_recursion(0)
0
>>> num_boxes_via_recursion(1)
1
>>> num_boxes_via_recursion(60)
2
```

In [52]:
import math
def num_boxes(n):
    '''
    n input for number of coins
    function will divide n by 30 and round upwards to give the minimum number of boxes n coins can be stored in
    if n is less than or equal to 0, returns 0
    '''
    if n<=0:
        return 0
    if n>0:
        return math.ceil(n/30)
print(num_boxes(31), num_boxes(1), num_boxes(0), num_boxes(60))

2 1 0 2


In [53]:
def num_boxes_via_recursion(n):
    '''
    n input for number of coins
    if n is less than or equal to 0, return 0
    else, return 1 and input the function again, subtracting 30 from n
    '''
    if n <= 0:
        return 0
    else:
        return 1 + num_boxes_via_recursion(n - 30)
print(num_boxes_via_recursion(31), num_boxes_via_recursion(1), num_boxes_via_recursion(0), num_boxes_via_recursion(60))

2 1 0 2


# Q2 — Simulation (Urns)

**Question:**

Urns  
A: 1W, 2B, 3R  
B: 2W, 1B, 1R  
C: 4W, 5B, 3R

One urn is chosen uniformly at random. Two balls are drawn **without replacement** and are observed to be **one white and one red**.  
Estimate \( P(\text{urn is B or C} \mid \text{draw is } \{W,R\}) \) by simulation and store the result in `est_prob`.

**Notes**
- Use Python’s `random` module.
- Use a **large** number of trials for a good estimate.
- Hint: after a `{W,R}` draw, record which urn produced it, then compute the conditional probability.

Solution 1:

In [ ]:
from fractions import Fraction

def P(event, space): 
    """
    The probability of an event, given a sample space of equiprobable outcomes.
    """
    return Fraction(len(event & space), 
                    len(space))

In [ ]:
def cross(A, B):
    """
    The set of ways of concatenating one item from A, a letter, with one from B, a number, so we can get, e.g. A1, B4, C5.
    """
    return {a + b 
            for a in A for b in B}

urn_a = cross('W', '1') | cross('B', '12') | cross('R', '123') 
urn_b = cross('W', '12') | cross('B', '1') | cross('R', '1') 
urn_c = cross('W', '1234') | cross('B', '12345') | cross('R', '123') 
total_urns = len([urn_a, urn_b, urn_c])

urn_a

{'B1', 'B2', 'R1', 'R2', 'R3', 'W1'}

In [ ]:
import itertools

def combos(items, n):
    """
    All combinations of n items; each combo as a concatenated str.
    n is a sample here.
    """
    return {' '.join(combo) 
            for combo in itertools.combinations(items, n)}

urna_2 = combos(urn_a, 2)

In [ ]:
# probability white and red ball from urn a
w1r1_urna = {s for s in urna_2 if s.count('W') == 1 and s.count('R') == 1}

P(w1r1_urna, urna_2)

Fraction(1, 5)

In [ ]:
# probability white and red ball from urn b
urnb_2 = combos(urn_b, 2)
w1r1_urnb = {s for s in urnb_2 if s.count('W') == 1 and s.count('R') == 1}

P(w1r1_urnb, urnb_2)

Fraction(1, 3)

In [ ]:
# probability white and red ball from urn c
urnc_2 = combos(urn_c, 2)
w1r1_urnc = {s for s in urnc_2 if s.count('W') == 1 and s.count('R') == 1}

P(w1r1_urnc, urnc_2)

Fraction(2, 11)

In [60]:
# probability of event (prob_event) meaning the probability any urn picks out a white and a red ball
prob_event = Fraction(1, 3) * (P(w1r1_urna, urna_2) + P(w1r1_urnb, urnb_2) + P(w1r1_urnc, urnc_2))
prob_event

Fraction(118, 495)

In [ ]:
prob_b_or_c_w1r1 = (((1/total_urns)*P(w1r1_urnb, urnb_2)) + ((1/total_urns)*P(w1r1_urnc, urnc_2)))/prob_event
est_prob = prob_b_or_c_w1r1
est_prob

0.7203389830508474

Solution 2

In [ ]:
urn_a = ["white", "black", "black", "red", "red", "red"]
urn_b = ["white", "white", "black", "red"]
urn_c = ["white", "white", "white", "white", "black", "black", "black", "black", "black", "red", "red", "red"]

urns = [urn_a, urn_b, urn_c]
no_urns = len(urns)

def prob_white_red(urn):
    total_pairs = math.comb(len(urn), 2)
    white_count = urn.count("white")
    red_count = urn.count("red")
    desired_outcome = white_count * red_count
    return desired_outcome / total_pairs

p_a = prob_white_red(urn_a)
p_b = prob_white_red(urn_b)
p_c = prob_white_red(urn_c)

# Prior probability of choosing each urn = 1/3
p_total = (p_a + p_b + p_c) / no_urns
p_b_or_c = ((p_b/no_urns) + (p_c/no_urns)) / p_total
est_prob = p_b_or_c

print("Exact probability (B or C | red+white):", est_prob)

Exact probability (B or C | red+white): 0.7203389830508475


Solution 3

In [69]:
import math
from collections import Counter

def prob_draw(urn, target_draw):
    """
    urn: list of ball colours (e.g., ["white","white","black","red"])
    target_draw: dict specifying desired outcome 
                 (e.g., {"white": 1, "red": 1})
    """
    total_balls = len(urn)
    total_ways = math.comb(total_balls, sum(target_draw.values()))
    
    # count how many of each colour in urn
    urn_counts = Counter(urn)
    
    # favourable ways = product of combinations for each colour
    favourable = 1
    for colour, count_needed in target_draw.items():
        favourable *= math.comb(urn_counts[colour], count_needed)
    
    return Fraction(favourable / total_ways).limit_denominator()
urn_a = ["white", "black", "black", "red", "red", "red"]
urn_b = ["white", "white", "black", "red"]
urn_c = ["white", "white", "white", "white",
         "black", "black", "black", "black", "black",
         "red", "red", "red"]

# Probability of 1 white + 1 red from each urn
prob_a = prob_draw(urn_a, {"white": 1, "red": 1})
prob_b = prob_draw(urn_b, {"white": 1, "red": 1})
prob_c = prob_draw(urn_c, {"white": 1, "red": 1})
print(prob_a)
print(prob_b)
print(prob_c)

1/5
1/3
2/11


In [70]:
def urn_or(prob_all, subset_indices):
    """
    prob_all: list of probabilities P(E|urn_i) for all urns
    subset_indices: indices of urns we care about (e.g. [1,2] for B or C)
    """
    n = len(prob_all)  # number of urns
    # total probability of event
    p_total = sum(p/n for p in prob_all)
    # probability of event AND urn in subset
    p_subset = sum(prob_all[i]/n for i in subset_indices)
    return p_subset / p_total

# Example: urns A, B, C in prob_all, we want B or C
est_prob = urn_or([prob_a, prob_b, prob_c], [1,2])
est_prob

Fraction(85, 118)

Solution 4

In [ ]:
import math

urn_a = ["white", "black", "black", "red", "red", "red"]
urn_b = ["white", "white", "black", "red"]
urn_c = ["white", "white", "white", "white", "black", "black", "black", "black", "black", "red", "red", "red"]

urns = [urn_a, urn_b, urn_c]
no_urns = len(urns)

def prob_white_red(urn):
    total_pairs = math.comb(len(urn), 2)
    white_count = urn.count("white")
    red_count = urn.count("red")
    desired_outcome = white_count * red_count
    return desired_outcome / total_pairs

p_a = prob_white_red(urn_a)
p_b = prob_white_red(urn_b)
p_c = prob_white_red(urn_c)

# Prior probability of choosing each urn = 1/3
p_total = (p_a + p_b + p_c) / no_urns
est_prob = ((p_b/no_urns) + (p_c/no_urns)) / p_total

print("Exact probability (B or C | red+white):", est_prob)

Exact probability (B or C | red+white): 0.7203389830508475
